# Hamilton MPE2

The MPE2 (Monitored Pressure Extractor) is Hamilton's positive pressure station for solid phase
extraction and filtration. A filter plate goes on its shuttle, with or without a collection plate
under it; the MPE2 clamps it under its manifold and pushes the liquid through with gas, into the
collection plate or into a waste container that its vacuum pump draws on.

| Property | Value |
|---|---|
| Communication | USB (virtual COM port, 921600 baud) or TCP (port 2000) |
| Protocol | Hamilton's HOI, as the Nimbus and the Prep |
| Units | pressure in psi, flow in standard L/min, heights in mm, times in s |

```{warning}
This driver has run on one MPE2 (firmware V1.9.5.619, over USB): queries, initialization, the
vacuum, the plate steps, processing into waste and into a collection plate. The reagent fill and
evaporator accessories aren't supported yet. Please report what works.
```

```{device-card} hamilton-mpe2
```

## How it communicates

The MPE2 speaks Hamilton's HOI protocol, the same as the Nimbus and the Prep, over a USB virtual
COM port or over TCP. The driver connects as a controller, sends each step of Hamilton's own
library (`HSLMPELib`) as the firmware takes it, and waits for long steps to finish by asking for
the last command's results. Firmware errors come back with Hamilton's own names and help text.

## Physical setup

- Gas (nitrogen or air) at the input. The firmware aborts a process step below 80 psi and above
  120 psi, and then has to be initialized again. The driver warns before clamping when the gas
  is low; the firmware's limits can't be changed.
- Keep the supply's rest under 120 psi: when the valve closes the supply overshoots by a few psi,
  and a supply resting at 118 psi aborts process steps. When the valve opens the supply dips:
  resting at about 114 psi, 30 psi through a filter plate took it to 84 psi, and 60 psi to about
  75 psi, which the tested MPE2 ran through without an abort; resting at 105 psi, 60 psi took it
  to 70 psi and aborted.
- A waste container connected; the vacuum pump draws the filtrate into it.
- No plates on the MPE2 before it initializes.
- USB: the MPE2 shows up as a COM port (an FTDI chip). Over the network: its address, port 2000.

## Connect

`MPE2()` finds the MPE2 on USB by its serial chip, on Windows, Linux and macOS. With several plugged
in, give one's USB serial number (`device_id="..."`), or the port (`port="COM4"`); on the network,
its host (`host="192.168.100.100"`). To try the driver without an MPE2, give it the simulator
instead: `MPE2(io=MPE2Simulator())`.

In [ ]:
from pylabrobot.hamilton.mpe2 import MPE2

mpe = MPE2()  # found on USB; or MPE2(device_id=...), MPE2(port=...), MPE2(host=...)
await mpe.setup()
print(mpe.firmware_version)

## Status and sensors

`request_status()` gives the plate state, the vacuum and the waste containers;
`request_pressures()` every pressure sensor, the gas supply among them.

In [ ]:
print(await mpe.request_status())
print(await mpe.request_pressures())

## Initialize

The motors home, and the vacuum runs to the first waste container in place. Take any plates off
first. Nothing is sent when the MPE2 is already initialized and ready.

In [ ]:
await mpe.initialize()

## The vacuum

The vacuum pump draws what passes the plate into the waste container. `run_for()` runs it for a
time and always stops it.

In [ ]:
await mpe.vacuum.run_for(3, waste_container=1)
print(await mpe.vacuum.request_is_on())

## Place a filter plate

Put a filter plate on the shuttle by hand (or with a robot), then tell the MPE2 its heights: from
the plate's top to its seat, and to the bottom of its nozzles. Nothing moves.

In [ ]:
await mpe.filter_plate_placed(height=14.4, nozzle_height=14.4)  # your plate's heights

## Clamp and retrieve

The shuttle carries the plate under the manifold, which clamps it (about 18 s); retrieving brings
it back (about 12 s). A step that is done already sends nothing.

In [ ]:
await mpe.clamp_filter_plate()

Bring it back out:

In [ ]:
await mpe.retrieve_filter_plate()

## Process into the waste

Control points are `PlateStep`s (type, value, seconds) for the whole plate (Hamilton's text,
`"pressure,5,2;pressure,15,2"`, reads in with `parse_control_points()`):
`pressure` in psi, `flow` in standard L/min, `idle`. The driver divides them by the manifold's
ports and checks them against the firmware's limits. The value asked is close to the manifold's
real pressure. Each control point lasts at least about 1 s, whatever is asked, and the gas stays
on about half a second longer than the points add up to.

In [ ]:
from pylabrobot.hamilton.mpe2 import PlateStep

steps = [PlateStep("pressure", 5, 2), PlateStep("pressure", 15, 2)]  # type, psi, seconds
await mpe.process_filter_to_waste_container(steps, waste_container=1, return_plate=True)

## Ramps, and recording the gas supply

When the valve opens at full pressure, the gas supply dips. `ramp()` rises in control points of at
least 1 s, which softens the dip (the supply still sags as far once at the level), and
`record_pressures()` reads the gas supply and the manifold about 14 times a second while a step
runs.

In [ ]:
from pylabrobot.hamilton.mpe2 import PlateStep, ramp

steps = ramp(to=30, duration=2) + [PlateStep("pressure", 30, 2)]
async with mpe.record_pressures() as recording:
  await mpe.process_filter_to_waste_container(steps, return_plate=True)
print(recording)  # readings, the lowest input and when, the manifold's peak

## Process into a collection plate

The order the MPE2 takes, as Hamilton's software runs it:

1. The filter plate placed and clamped up.
2. The collection plate put on the shuttle, and `collection_plate_placed()` (nothing moves).
3. `retrieve_filter_plate()` lowers the filter plate onto the collection plate and brings the two
   out together, to pipette into.
4. `process_filter_to_collection_plate()` clamps the pair, pushes the liquid through, and brings them
   back out.
5. `clamp_filter_plate()` lifts the filter plate off the collection plate: the MPE2 takes the
   collection plate off only then.
6. The collection plate taken off, `collection_plate_removed()`; `retrieve_filter_plate()`; the
   filter plate taken off, `filter_plate_removed()`.

In [ ]:
await mpe.filter_plate_placed(height=14.4, nozzle_height=14.4)
await mpe.clamp_filter_plate()

Put the collection plate on the shuttle, under the clamped filter plate, then tell the MPE2
(nothing moves) and bring the two out together:

In [ ]:
await mpe.collection_plate_placed(height=14.4, offset_from_nozzles=0.0)  # your plate's
await mpe.retrieve_filter_plate()  # the two plates out, to pipette into

Pipette into the filter plate, then push it through into the collection plate:

In [ ]:
await mpe.process_filter_to_collection_plate([PlateStep("pressure", 10, 3)], return_plate=True)

To take the plates off, first lift the filter plate off the collection plate:

In [ ]:
await mpe.clamp_filter_plate()  # the filter plate up, off the collection plate

Take the collection plate off the shuttle first: the filter plate then comes down where it stood.

In [ ]:
await mpe.collection_plate_removed()
await mpe.retrieve_filter_plate()  # the filter plate out

Take the filter plate off, then:

In [ ]:
await mpe.filter_plate_removed()

## When something goes wrong

What happens after an error depends on its kind:

- A pressure error (the gas supply, valves, manifold, vacuum, waste) recovers by itself: the
  running command aborted, the vacuum stopped, the filter plate brought back. The error says
  what was done: `err.recovery`.
- An intrusion, a motor error, a timeout or an unknown error stops everything that runs, moves
  nothing, and holds: every step that moves is refused until someone has looked at the MPE2 and
  called `recover()`.

`halt()` stops everything that runs at any time and moves nothing; `recover()` also brings the
filter plate back (this moves).

In [ ]:
report = await mpe.recover()
print(report)

## Close

`stop()` stops everything that runs (a running command, the vacuum pump), moves
nothing, and closes the connection.

In [ ]:
await mpe.stop()

## Acknowledgements

This driver was developed and tested at DALSA, the DTU Arena for Life Science Automation.